# 🎬 HindiDub server on a free Kaggle GPU

This notebook runs the HindiDub backend on Kaggle's free GPU and exposes it at your **fixed ngrok URL**,
so the Android app always finds it.

**One-time setup** (in this notebook's editor):
1. **Settings → Accelerator → GPU T4 x2**
2. **Settings → Internet → On** (needs a phone-verified Kaggle account)
3. **Add-ons → Secrets**, add and tick (attach) these:
   - `NGROK_AUTHTOKEN`: from https://dashboard.ngrok.com/get-started/your-authtoken
   - `NGROK_DOMAIN`: your free static domain from https://dashboard.ngrok.com/domains (e.g. `abc-xyz.ngrok-free.app`)
   - `HINDIDUB_API_KEY`: any long random password (the same value goes into GitHub secrets)

**Every time you want to dub:** click **Save Version → Save & Run All (Commit)**.
It runs in the background (you can close the browser) for up to 12 h, and **stops by itself after 45 min with no work**
so it doesn't waste your 30 free GPU hours a week.

In [ ]:
# ---- settings ----
REPO = "https://github.com/Gourav-Chandila/Hindi-Dubber.git"
BRANCH = "feature/hindi-dub-app"   # change to "main" after merging
IDLE_STOP_MINUTES = 45             # stop when nothing happened for this long
MAX_HOURS = 11.5                   # Kaggle hard limit is 12 h

from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
def secret(name, required=True):
    try:
        return _s.get_secret(name)
    except Exception:
        if required:
            raise RuntimeError(f"Missing Kaggle secret {name}: add it under Add-ons -> Secrets and tick it.")
        return ""

NGROK_AUTHTOKEN = secret("NGROK_AUTHTOKEN")
NGROK_DOMAIN = secret("NGROK_DOMAIN").replace("https://", "").strip("/")
API_KEY = secret("HINDIDUB_API_KEY", required=False)
print("ngrok domain:", NGROK_DOMAIN, "| API key set:", bool(API_KEY))

In [ ]:
# ---- install (about 5-8 minutes) ----
import subprocess, sys, os, glob
def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
sh("apt-get -qq update && apt-get -qq install -y ffmpeg rubberband-cli > /dev/null")
sh(f"rm -rf /tmp/hindidub-src && git clone -q --depth 1 -b {BRANCH} {REPO} /tmp/hindidub-src")
sh("pip -q install uv pyngrok")
# Separate Python 3.11 env so Kaggle's preinstalled packages never conflict.
sh("uv venv -q --allow-existing --python 3.11 /tmp/venv")  # safe to re-run
PY = "/tmp/venv/bin/python"
sh(f"uv pip install -q --python {PY} torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121")
sh(f"uv pip install -q --python {PY} -r /tmp/hindidub-src/backend/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps demucs==4.0.1")
sh(f"uv pip install -q --python {PY} dora-search==0.1.12 einops julius==0.2.7 lameenc openunmix==1.3.0 pyyaml tqdm")

# CUDA libs shipped as pip wheels (cuDNN/cuBLAS) for faster-whisper.
nvidia_libs = glob.glob("/tmp/venv/lib/python3.11/site-packages/nvidia/*/lib")
ENV = dict(os.environ,
    HD_DEVICE="cuda", HD_DATA_DIR="/tmp/hindidub", HD_API_KEY=API_KEY,
    COQUI_TOS_AGREED="1", HF_HOME="/tmp/hf", TORCH_HOME="/tmp/torch",
    MPLBACKEND="Agg",  # Kaggle's inline plot backend doesn't exist inside our venv
    LD_LIBRARY_PATH=":".join(nvidia_libs + [os.environ.get("LD_LIBRARY_PATH", "")]),
)
sh(f"{PY} -c 'import torch; print(\"torch\", torch.__version__, \"cuda:\", torch.cuda.is_available())'")

In [ ]:
# ---- download all models once (about 3-5 minutes) ----
subprocess.run([PY, "scripts/prefetch_models.py"], cwd="/tmp/hindidub-src/backend", env=ENV, check=True)

In [ ]:
# ---- start server + public URL ----
import time, requests
log = open("/tmp/server.log", "w")
server = subprocess.Popen(
    ["/tmp/venv/bin/uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "7860", "--timeout-keep-alive", "120"],
    cwd="/tmp/hindidub-src/backend", env=ENV, stdout=log, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(requests.get("http://127.0.0.1:7860/health", timeout=5).json()); break
    except Exception:
        time.sleep(2)
else:
    print(open("/tmp/server.log").read()[-3000:]); raise RuntimeError("server did not start")

# Public URL via ngrok (logic lives in the repo so fixes arrive with git pull)
exec(open("/tmp/hindidub-src/notebooks/tunnel.py").read())

In [ ]:
# ---- keep running until idle (the phone app talks to the server meanwhile) ----
start = time.time()
while True:
    time.sleep(60)
    if server.poll() is not None:
        print(open("/tmp/server.log").read()[-5000:]); raise RuntimeError("server crashed")
    try:
        h = requests.get("http://127.0.0.1:7860/health", timeout=10).json()
    except Exception as e:
        print("health check failed:", e); continue
    up = (time.time() - start) / 3600
    print(f"[{up:4.1f} h] busy={h['busy']} queued={h['queued']} idle={h['idle_seconds']//60} min", flush=True)
    if not h["busy"] and h["queued"] == 0 and h["idle_seconds"] > IDLE_STOP_MINUTES * 60:
        print("Idle - stopping to save GPU quota."); break
    if up > MAX_HOURS:
        print("Reached Kaggle time limit."); break

ngrok_proc.terminate(); server.terminate()
print(open("/tmp/server.log").read()[-3000:])